# T16 — Estado dos 20 modelos e comparação por datas comuns

Cada MAE abaixo vem do teste salvo no próprio notebook do modelo. **Os valores individuais não formam um ranking:** a comparação final de uma base usa somente a interseção das datas em que seus quatro modelos fizeram previsão. Ter o mesmo número de previsões não demonstra que as datas são iguais. MAEs de bases diferentes também não são comparados entre si.

In [1]:
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
from IPython.display import display

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'preparacao_bases.py').is_file())
BASES = {1: 'bitcoin', 2: 'trafego', 3: 'poluicao', 4: 'clima', 5: 'ouro'}
ARQUIVOS = {'Random Forest': 'random-forest', 'Holt-Winters': 'holt-winters',
            'SARIMAX': 'sarimax', 'MLP Regressor': 'mlp-regressor'}

def saidas_e_execucao(caminho):
    nb = json.loads(caminho.read_text(encoding='utf-8'))
    codigos = [c for c in nb['cells'] if c['cell_type'] == 'code']
    textos = []
    for celula in codigos:
        for saida in celula.get('outputs', []):
            textos.append(''.join(saida.get('text', [])))
            textos.append(''.join(saida.get('data', {}).get('text/plain', [])))
    return '\n'.join(textos), sum(c.get('execution_count') is not None for c in codigos), len(codigos)

def resultado_individual(grupo, modelo):
    caminho = RAIZ / f'grupo{grupo}' / f'{ARQUIVOS[modelo]}-{grupo}.ipynb'
    linha = {'grupo': grupo, 'base': BASES[grupo], 'modelo': modelo,
             'MAE_teste_individual': np.nan, 'n_teste': pd.NA,
             'estado': 'notebook ausente', 'origem': ''}
    if not caminho.exists():
        return linha
    linha['origem'] = str(caminho.relative_to(RAIZ))
    texto, executadas, total = saidas_e_execucao(caminho)
    linha['estado'] = f'{executadas}/{total} células executadas'
    if modelo == 'Holt-Winters':
        if grupo == 1:
            linha['estado'] += '; protótipo mensal incompatível com o teste diário'
        return linha
    if modelo == 'Random Forest':
        if grupo == 4:
            achado = re.search(r'^\s*0\s+Random Forest\s+([\d,]+)\s+([\d.]+)', texto, re.M)
            if achado: n, mae = achado.groups()
        else:
            mae_encontrado = re.search(r'MAE RF:\s*([\d.]+)', texto)
            n_encontrado = re.search(r'previsões:\s*([\d,]+)', texto)
            achado = bool(mae_encontrado and n_encontrado)
            if achado: mae, n = mae_encontrado.group(1), n_encontrado.group(1)
    elif modelo == 'MLP Regressor':
        achado = re.search(r'teste_final\s+([\d,]+)\s+([\d.]+)', texto)
        if achado: n, mae = achado.groups()
    else:
        mae_encontrado = re.search(r'MAE de teste:\s*([\d.]+)', texto)
        n_encontrado = re.search(r'Previsões de um(?:a)? [^:]+:\s*([\d,]+)', texto)
        achado = bool(mae_encontrado and n_encontrado)
        if achado: mae, n = mae_encontrado.group(1), n_encontrado.group(1)
    if achado:
        linha['MAE_teste_individual'] = float(mae)
        linha['n_teste'] = int(n.replace(',', ''))
        if executadas < total:
            linha['estado'] += '; execução parcial — conferir reprodução'
        if modelo == 'MLP Regressor' and grupo in (1, 5):
            linha['estado'] += '; revisão após ver teste — MAE exploratório'
    return linha

resultados = pd.DataFrame([
    resultado_individual(grupo, modelo)
    for grupo in BASES for modelo in ARQUIVOS
])
display(resultados.round(4))
print(f'MAEs individuais disponíveis: {resultados.MAE_teste_individual.notna().sum()}/20')

,grupo,base,modelo,MAE_teste_individual,n_teste,estado,origem
0,1,bitcoin,Random Forest,1661.4218,884,9/9 células executadas,grupo1\random-forest-1.ipynb
1,1,bitcoin,Holt-Winters,NaN,<NA>,15/15 células executadas; protótipo mensal inc...,grupo1\holt-winters-1.ipynb
2,1,bitcoin,SARIMAX,1391.9008,884,22/22 células executadas,grupo1\sarimax-1.ipynb
3,1,bitcoin,MLP Regressor,1401.8277,884,17/17 células executadas; revisão após ver tes...,grupo1\mlp-regressor-1.ipynb
4,2,trafego,Random Forest,153.2877,10267,12/12 células executadas,grupo2\random-forest-2.ipynb
5,2,trafego,Holt-Winters,NaN,<NA>,notebook ausente,
6,2,trafego,SARIMAX,NaN,<NA>,10/15 células executadas,grupo2\sarimax-2.ipynb
7,2,trafego,MLP Regressor,180.4071,10267,17/17 células executadas,grupo2\mlp-regressor-2.ipynb
8,3,poluicao,Random Forest,9.8670,6336,13/13 células executadas,grupo3\random-forest-3.ipynb
9,3,poluicao,Holt-Winters,NaN,<NA>,notebook ausente,


MAEs individuais disponíveis: 13/20


## Comparação oficial na interseção

O bloco abaixo aceita previsões completas incorporadas aos próprios notebooks, com datas, valores reais e previstos. Faz `inner join` das quatro séries por data, verifica que o valor real coincide e recalcula cada MAE na mesma interseção. Se faltar um modelo ou suas previsões completas, o grupo fica pendente. Os MAEs individuais acima não entram automaticamente.

In [2]:
TIPO_PREVISOES = 'application/vnd.series-temporais.previsoes+json'

def previsoes_do_notebook(grupo, modelo):
    caminho = RAIZ / f'grupo{grupo}' / f'{ARQUIVOS[modelo]}-{grupo}.ipynb'
    if not caminho.exists():
        return None
    nb = json.loads(caminho.read_text(encoding='utf-8'))
    cargas = [saida.get('data', {}).get(TIPO_PREVISOES)
              for celula in nb['cells'] for saida in celula.get('outputs', [])]
    cargas = [c for c in cargas if c is not None]
    if not cargas:
        return None
    if len(cargas) > 1:
        raise ValueError(f'{caminho}: mais de uma série completa de previsões')
    dados = pd.DataFrame(cargas[0])
    if set(dados.columns) != {'data', 'real', 'previsto'}:
        raise ValueError(f'{caminho}: colunas de previsão inválidas')
    dados['data'] = pd.to_datetime(dados['data'])
    dados = dados.set_index('data').sort_index()
    if dados.index.has_duplicates or dados.isna().any().any():
        raise ValueError(f'{caminho}: datas duplicadas ou valores faltantes')
    return dados

comparacoes = []
pendencias = []
for grupo in BASES:
    series = {modelo: previsoes_do_notebook(grupo, modelo) for modelo in ARQUIVOS}
    faltantes = [modelo for modelo, serie in series.items() if serie is None]
    if faltantes:
        pendencias.append({'grupo': grupo, 'base': BASES[grupo],
                           'pendencia': 'previsões completas: ' + ', '.join(faltantes)})
        continue
    datas_comuns = series[next(iter(series))].index
    for serie in series.values():
        datas_comuns = datas_comuns.intersection(serie.index)
    if datas_comuns.empty:
        raise ValueError(f'Grupo {grupo}: interseção de datas vazia')
    referencia = series[next(iter(series))].loc[datas_comuns, 'real'].to_numpy()
    for modelo, serie in series.items():
        recorte = serie.loc[datas_comuns]
        np.testing.assert_allclose(recorte.real.to_numpy(), referencia,
                                   err_msg=f'Grupo {grupo}, {modelo}: alvo divergente')
        comparacoes.append({'grupo': grupo, 'base': BASES[grupo], 'modelo': modelo,
                            'n_datas_comuns': len(datas_comuns),
                            'inicio': datas_comuns.min(), 'fim': datas_comuns.max(),
                            'MAE_na_intersecao': np.abs(referencia - recorte.previsto.to_numpy()).mean()})
if pendencias:
    display(pd.DataFrame(pendencias))
if comparacoes:
    comparacao = pd.DataFrame(comparacoes)
    comparacao['posicao_na_base'] = comparacao.groupby('grupo').MAE_na_intersecao.rank(method='min')
    display(comparacao.sort_values(['grupo', 'posicao_na_base']).round(4))
else:
    print('Ranking oficial pendente: nenhuma base possui as quatro séries completas de previsão.')

,grupo,base,pendencia
0,1,bitcoin,"previsões completas: Random Forest, Holt-Winte..."
1,2,trafego,"previsões completas: Random Forest, Holt-Winte..."
2,3,poluicao,"previsões completas: Random Forest, Holt-Winte..."
3,4,clima,"previsões completas: Random Forest, Holt-Winte..."
4,5,ouro,"previsões completas: Random Forest, Holt-Winte..."


Ranking oficial pendente: nenhuma base possui as quatro séries completas de previsão.


### Próximas entregas

- Alinhar e executar Holt-Winters na granularidade oficial de cada base.
- Concluir SARIMAX 2 e 3 sem tratar horas separadas como instantes consecutivos; reproduzir integralmente o notebook 5.
- Incorporar as quatro séries de previsões por base no formato aceito acima e então calcular o ranking sobre as datas comuns.
- Interpretar o desempenho diante da persistência e os erros por período em cada base.